In [ ]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader

### load all txt files in the directory
dir_loader = DirectoryLoader(
    "../data/dukcapil_pdf", 
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False)

pdf_docs = dir_loader.load()
pdf_docs

In [ ]:
print(f"Total pages extracted: {len(pdf_docs)}")
print(f"Page 0: {pdf_docs[0].page_content[:100]}")
print(f"Page 1: {pdf_docs[1].page_content[:100]}")
print(f"Page 5: {pdf_docs[5].page_content[:100]}")

In [ ]:
import re

for doc in pdf_docs:
    page_num = doc.metadata['page']
    text = doc.page_content
    
    # hitung jumlah karakter total
    total_chars = len(text)
    
    # hitung karakter yang "normal" (huruf, angka, spasi)
    normal_chars = len(re.findall(r'[a-zA-Z0-9\s]', text))
    
    # hitung rasio karakter normal
    ratio = normal_chars / total_chars if total_chars > 0 else 0
    
    print(f"Halaman {page_num} | Total chars: {total_chars} | Ratio normal: {ratio:.2f}")

## Step 1 — Filter Halaman

Buang halaman yang tidak berguna untuk RAG:
- **Halaman 0 & 281**: cover scan (sampah)
- **Halaman 3–24**: daftar isi (penuh titik-titik & nomor halaman, tidak ada nilai konten)

Opsional dibuang: halaman 1–2 (kata pengantar) — tergantung apakah relevan untuk pertanyaan user.

In [ ]:
COVER_PAGES = {0, 281}          # scan sampah
DAFTAR_ISI_PAGES = set(range(3, 25))  # page 3–24 (0-indexed), ratio rendah

SKIP_PAGES = COVER_PAGES | DAFTAR_ISI_PAGES

filtered_docs = [doc for doc in pdf_docs if doc.metadata['page'] not in SKIP_PAGES]

print(f"Sebelum filter: {len(pdf_docs)} halaman")
print(f"Sesudah filter : {len(filtered_docs)} halaman")
print(f"Dibuang        : {sorted(SKIP_PAGES)}")

## Step 2 — Clean Teks per Halaman

Masalah yang perlu diatasi:
1. Sisa titik-titik daftar isi yang bisa bocor ke halaman awal (`......`)
2. Nomor halaman standalone di akhir baris (mis. `\n18\n`)
3. Whitespace berlebihan (tab, spasi ganda, baris kosong berulang)
4. Newline di tengah kalimat karena layout dua kolom / word-wrap PDF

In [ ]:
import re
import copy

def clean_page(text: str) -> str:
    # 1. Hapus deretan titik (artefak daftar isi)
    text = re.sub(r'\.{3,}', '', text)
    
    # 2. Hapus nomor halaman standalone (baris hanya berisi angka)
    text = re.sub(r'^\s*\d{1,3}\s*$', '', text, flags=re.MULTILINE)
    
    # 3. Gabungkan baris yang terpotong di tengah kalimat:
    #    baris yang berakhir bukan dengan tanda baca → sambung dengan spasi
    text = re.sub(r'(?<![.!?:\-])\n(?=[a-zA-Z])', ' ', text)
    
    # 4. Normalisasi whitespace
    text = re.sub(r'[ \t]+', ' ', text)          # spasi ganda → satu
    text = re.sub(r'\n{3,}', '\n\n', text)       # baris kosong berulang → maks 2
    
    return text.strip()


cleaned_docs = []
for doc in filtered_docs:
    new_doc = copy.copy(doc)
    new_doc.page_content = clean_page(doc.page_content)
    if new_doc.page_content:   # buang kalau setelah cleaning jadi kosong
        cleaned_docs.append(new_doc)

print(f"Dokumen setelah cleaning: {len(cleaned_docs)} halaman")

# Spot-check
for page_idx in [1, 25, 26, 100, 278]:
    candidates = [d for d in cleaned_docs if d.metadata['page'] == page_idx]
    if candidates:
        print(f"\n=== HALAMAN {page_idx} (setelah cleaning) ===")
        print(candidates[0].page_content[:400])

## Step 3 — Verifikasi Kualitas Final

In [ ]:
import statistics

lengths = [len(d.page_content) for d in cleaned_docs]
ratios = [len(re.findall(r'[a-zA-Z0-9\s]', d.page_content)) / len(d.page_content)
          for d in cleaned_docs if len(d.page_content) > 0]

print(f"Total halaman  : {len(cleaned_docs)}")
print(f"Chars rata-rata: {statistics.mean(lengths):.0f}")
print(f"Chars min/max  : {min(lengths)} / {max(lengths)}")
print(f"Ratio bersih   : min={min(ratios):.2f}  mean={statistics.mean(ratios):.2f}  max={max(ratios):.2f}")

# Flagging halaman yang masih mencurigakan (ratio < 0.85)
suspicious = [(d.metadata['page'], len(d.page_content),
               len(re.findall(r'[a-zA-Z0-9\s]', d.page_content)) / len(d.page_content))
              for d in cleaned_docs if len(d.page_content) > 0
              and len(re.findall(r'[a-zA-Z0-9\s]', d.page_content)) / len(d.page_content) < 0.85]

if suspicious:
    print(f"\nHalaman masih mencurigakan (ratio < 0.85):")
    for page, chars, ratio in suspicious:
        print(f"  Halaman {page:3d} | chars={chars:4d} | ratio={ratio:.2f}")
else:
    print("\nSemua halaman bersih (ratio >= 0.85)")

## Step 4 — Tag Section Metadata

Tag setiap halaman dengan section asal (BAB I/II/III/Kata Pengantar). Metadata ini akan dipakai chunker di tahap berikutnya untuk strategi chunking yang berbeda per section (Q&A-aware untuk BAB II, character-based untuk BAB I/III).

In [ ]:
def tag_section(page: int) -> str:
    if page in (1, 2):
        return "Kata Pengantar"
    if 25 <= page <= 31:
        return "BAB I - Pendahuluan"
    if 32 <= page <= 277:
        return "BAB II - Pertanyaan dan Jawaban"
    if 278 <= page <= 280:
        return "BAB III - Penutup"
    return "Unknown"

for doc in cleaned_docs:
    doc.metadata["section"] = tag_section(doc.metadata["page"])

# Distribusi section
from collections import Counter
section_counts = Counter(d.metadata["section"] for d in cleaned_docs)
for section, count in section_counts.items():
    print(f"{section:40s} : {count:3d} halaman")

## Step 5 — Export ke Pickle

Dump `cleaned_docs` ke pickle supaya bisa dipakai langsung di `build_vectorstore.ipynb` tanpa perlu rerun preprocessing dari awal.

In [ ]:
import pickle
from pathlib import Path

OUTPUT_PATH = Path("../data/cleaned_docs.pkl")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_PATH, "wb") as f:
    pickle.dump(cleaned_docs, f)

print(f"Saved {len(cleaned_docs)} docs to {OUTPUT_PATH.resolve()}")

## Step 6 — Export ke JSON (untuk inspeksi manual)

Dump versi JSON yang isinya **identik** dengan pickle (list of `{page_content, metadata}`). Gunanya untuk debugging/inspect tanpa harus unpickle di Python.

In [ ]:
import json

JSON_PATH = Path("../data/cleaned_docs.json")
with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(
        [{"page_content": d.page_content, "metadata": d.metadata} for d in cleaned_docs],
        f, ensure_ascii=False, indent=2,
    )
print(f"Saved {len(cleaned_docs)} docs → {JSON_PATH.resolve()}")

# Verifikasi PKL == JSON (round-trip)
with open(JSON_PATH, "r", encoding="utf-8") as f:
    json_docs = json.load(f)
assert len(json_docs) == len(cleaned_docs), "Count mismatch"
for i, (d, j) in enumerate(zip(cleaned_docs, json_docs)):
    assert d.page_content == j["page_content"], f"page_content mismatch at idx {i}"
    assert d.metadata == j["metadata"], f"metadata mismatch at idx {i}"
print("✓ PKL ↔ JSON content identical")